### Here we will analyze the user journeys, defining what a journey is. If we want to analyze a listening of a tour, we should be taking acount a specific session of a user. A user might listen to a tour a different date so we need to be able to define when a user "journey" begins and ends to get specific info about this. 

### In the previous notebooks:
1) We analyzed the robustness of the pseudo user id finding that we will use that id for user authentication
2) We found the correct story order of the tours from which we can analyze how much the users follow the correct path when listening to a tour 
3) We found that IOS data cannot be used to provide clear insights without altering the data based on assumptions that are not 100% robust (We dont know for sure when a story begun and ended).

In [1]:
import gc
import pandas as pd
from pathlib import Path

In [ ]:
# del story_events
# gc.collect()

In [2]:
BASE_DIR = Path.cwd().parent
events_path = BASE_DIR / "data/clean/events.parquet"

In [3]:
events_data = pd.read_parquet(events_path)

In [4]:
listening_events = events_data.loc[~events_data.tour_id.isna()]

In [6]:
ANDROID = listening_events[listening_events['platform'] == "ANDROID"]

## 1) Defining user journeys.

lets check if the same user_pseudo_id listened to the same tour_id on multiple dates.

In [15]:
ANDROID.groupby("event_name", observed=True).size().sort_values(ascending=False)

event_name
tour_download_progress    1216412
story_start                696476
story_listened_20          489766
story_listened_40          444479
story_listened_60          412362
story_listened_80          385498
story_completed            350942
click_item                 263233
screen_view                232970
collapse_player            142253
pause                      120313
play                        67718
click_story                 39600
start_tour                  38195
forward_10                  33652
click_progress_bar          27353
backward_10                 25019
click_listen_now            22201
expand_player               21842
next_story                  10162
previous_story               6828
click_more                   6405
click_back                   5174
tour_download_started        4032
click_download_tour          4027
click_location               3630
tour_download_ended          3227
click_3d_map                 1831
click_copy_ref_code           890
cli

In [27]:
listening_events = [
    "start_tour",
    "story_start",
    "story_listened_20",
    "story_listened_40",
    "story_listened_60",
    "story_listened_80",
    "story_completed",
    "play",
    "pause",
    "forward_10",
    "backward_10",
    "click_progress_bar",
    "next_story",
    "previous_story",
    "click_story",
    # "click_item",
    # "click_listen_now"
]

ANDROID_LISTEN = ANDROID[ANDROID["event_name"].isin(listening_events)]

In [28]:
df = ANDROID_LISTEN[["user_pseudo_id", "tour_id", "event_datetime"]].dropna(subset=["tour_id"]).copy()

df["listen_date"] = pd.to_datetime(df["event_datetime"]).dt.date

tour_days = (
    df.groupby(["user_pseudo_id", "tour_id"])["listen_date"]
      .nunique()
      .reset_index(name="distinct_days")
)

multi_day_tours = tour_days[tour_days["distinct_days"] > 1]

print("user-tour pairs total:", len(tour_days))
print("user-tour pairs on multiple days:", len(multi_day_tours))
print("percentage:", len(multi_day_tours) / len(tour_days))

multi_day_tours.head(20)

user-tour pairs total: 9779
user-tour pairs on multiple days: 2039
percentage: 0.2085080274056652


,user_pseudo_id,tour_id,distinct_days
2,000b52af1c5a69eb96f0ef158a18dbe0,535,2
6,003cde637b910a8e8c5e911db3ef41a6,820,2
11,00810b9e3c1156eddc1326d14cc6ca64,869,2
18,00bee13b161139ae8887bdc825a511e0,535,2
20,00c9f1dd8dc43f45b70bef081b0b8571,865,2
21,00d1ca3ccdc87db57b02f3925913f2b7,107,3
23,00d2c86db17217114ba59dac500cf34b,858,2
29,01099eb2970b85d4eb3cc6c34c628800,539,2
30,010cccb789f7ea4ebc32f50ceb9ed292,873,2
32,01245714bf91443f88d5d5997a32e7d6,240,2


In [29]:
examples = multi_day_tours.head(20)[["user_pseudo_id", "tour_id"]]

example_rows = df.merge(examples, on=["user_pseudo_id", "tour_id"], how="inner")

example_rows = example_rows.sort_values(["user_pseudo_id", "tour_id", "event_datetime"])
example_rows[["user_pseudo_id", "tour_id", "event_datetime", "listen_date"]]

,user_pseudo_id,tour_id,event_datetime,listen_date
6934,000b52af1c5a69eb96f0ef158a18dbe0,535,2025-10-19 12:49:15.358000,2025-10-19
6935,000b52af1c5a69eb96f0ef158a18dbe0,535,2025-10-19 12:49:16.175002,2025-10-19
6936,000b52af1c5a69eb96f0ef158a18dbe0,535,2025-10-19 12:49:32.041004,2025-10-19
6937,000b52af1c5a69eb96f0ef158a18dbe0,535,2025-10-19 12:49:47.082005,2025-10-19
6938,000b52af1c5a69eb96f0ef158a18dbe0,535,2025-10-19 12:50:02.433006,2025-10-19
...,...,...,...,...
6929,02a20a988ac085d587ec933c5a5269fc,539,2025-10-16 11:07:04.159060,2025-10-16
6930,02a20a988ac085d587ec933c5a5269fc,539,2025-10-16 11:07:04.706061,2025-10-16
6931,02a20a988ac085d587ec933c5a5269fc,539,2025-10-16 11:07:18.857062,2025-10-16
6932,02a20a988ac085d587ec933c5a5269fc,539,2025-10-16 11:07:33.084063,2025-10-16


In [19]:
weird_guy = ANDROID[ANDROID['user_pseudo_id'] == "605230c80228394702c071059071ca39"]

In [22]:
ANDROID.loc[
    (ANDROID['tour_id'] == 820) &
    (ANDROID['language'] == "it-it") &
    (ANDROID['event_datetime'].dt.date == pd.to_datetime("2025-07-30").date())
]

,event_datetime,event_name,platform,language,user_id,user_pseudo_id,tour_id,story_id,lang_id,audio_time_played,audio_time_paused
1225465,2025-07-30 16:16:56.709000,start_tour,ANDROID,it-it,<NA>,003cde637b910a8e8c5e911db3ef41a6,820,<NA>,4,NaN,NaN
510895,2025-07-30 16:16:56.957001,story_start,ANDROID,it-it,<NA>,003cde637b910a8e8c5e911db3ef41a6,820,50562,4,NaN,NaN
1225466,2025-07-30 16:16:56.958002,collapse_player,ANDROID,it-it,<NA>,003cde637b910a8e8c5e911db3ef41a6,820,50562,4,NaN,NaN
1494515,2025-07-30 16:17:05.212003,click_item,ANDROID,it-it,<NA>,003cde637b910a8e8c5e911db3ef41a6,820,<NA>,4,NaN,NaN
956182,2025-07-30 16:17:05.212004,screen_view,ANDROID,it-it,<NA>,003cde637b910a8e8c5e911db3ef41a6,820,<NA>,4,NaN,NaN
1494514,2025-07-30 16:17:05.262005,click_item,ANDROID,it-it,<NA>,003cde637b910a8e8c5e911db3ef41a6,820,<NA>,4,NaN,NaN
242330,2025-07-30 16:17:05.262006,screen_view,ANDROID,it-it,<NA>,003cde637b910a8e8c5e911db3ef41a6,820,<NA>,4,NaN,NaN
242331,2025-07-30 16:17:07.480007,pause,ANDROID,it-it,<NA>,003cde637b910a8e8c5e911db3ef41a6,820,50562,4,NaN,00:09
519768,2025-07-30 16:19:22.090008,start_tour,ANDROID,it-it,<NA>,003cde637b910a8e8c5e911db3ef41a6,820,<NA>,4,NaN,NaN
1585158,2025-07-30 16:19:22.275010,story_start,ANDROID,it-it,<NA>,003cde637b910a8e8c5e911db3ef41a6,820,50562,4,NaN,NaN


In [ ]:
events_data.loc[
    (events_data["event_name"] == 'start_tour') & (events_data["tour_id"].isna())
]

### Big problem: There is a instrumentation artifact meaning we have story start, listened_20,40,60,80 and completion in the SAME second. Probably caused: When the user opened the story at a point where it was already finished / cached / auto-resumed, the app likely triggered all progress milestones instantly.

Lets find these instances to see how big is this problem

In [8]:
events = [
    "story_start",
    "story_listened_20",
    "story_listened_40",
    "story_listened_60",
    "story_listened_80",
    "story_completed"
]

df = ANDROID[ANDROID["event_name"].isin(events)].copy()

# round timestamp to second
df["second"] = df["event_datetime"].dt.floor("s")

# count events per story per second
same_second = (
    df.groupby(["user_pseudo_id","tour_id","story_id","second"])
      .size()
      .reset_index(name="event_count")
)

# keep suspicious ones
same_second = same_second[same_second["event_count"] >= 3]

In [9]:
inspect = df.merge(
    same_second[["user_pseudo_id", "tour_id", "story_id", "second"]],
    on=["user_pseudo_id", "tour_id", "story_id", "second"],
    how="inner"
)

inspect = inspect.sort_values(["user_pseudo_id", "tour_id", "story_id", "event_datetime"])

In [10]:
inspect.shape

(688279, 12)

In [41]:
inspect.event_name.unique()

<ArrowStringArray>
[ 'story_listened_40',  'story_listened_60',        'story_start',
  'story_listened_20',  'story_listened_80',    'story_completed',
        'backward_10',         'forward_10',        'click_story',
    'collapse_player',              'pause', 'click_progress_bar',
               'play',         'next_story',      'expand_player',
     'previous_story']
Length: 16, dtype: str

In [38]:
ANDROID.shape

(5076990, 11)

In [39]:
df.shape

(2779523, 12)

### This data cannot be trusted. I dont know why exactly these triggers are happening, but they most certainly inflate the numbers of the stories completed and listened. I will drop these and then run again the same metrics to see how much they change.

In [12]:
keys = ["user_pseudo_id", "tour_id", "story_id", "event_datetime", "event_name"]

ANDROID_clean = ANDROID.merge(
    inspect[keys],
    on=keys,
    how="left",
    indicator=True
)

ANDROID_clean = ANDROID_clean[ANDROID_clean["_merge"] == "left_only"].drop(columns="_merge")

In [13]:
ANDROID_clean.shape

(4388711, 11)

In [14]:
ANDROID_clean.groupby("event_name", observed=True).size().sort_values(ascending=False)

event_name
tour_download_progress    1216412
story_start                587116
story_listened_20          342069
story_listened_40          310528
story_listened_60          295326
story_listened_80          287170
story_completed            269035
click_item                 263233
screen_view                232970
collapse_player            142253
pause                      120313
play                        67718
click_story                 39600
start_tour                  38195
forward_10                  33652
click_progress_bar          27353
backward_10                 25019
click_listen_now            22201
expand_player               21842
next_story                  10162
previous_story               6828
click_more                   6405
click_back                   5174
tour_download_started        4032
click_download_tour          4027
click_location               3630
tour_download_ended          3227
click_3d_map                 1831
click_copy_ref_code           890
cli

In [15]:
ANDROID.groupby("event_name", observed=True).size().sort_values(ascending=False)

event_name
tour_download_progress    1216412
story_start                696476
story_listened_20          489766
story_listened_40          444479
story_listened_60          412362
story_listened_80          385498
story_completed            350942
click_item                 263233
screen_view                232970
collapse_player            142253
pause                      120313
play                        67718
click_story                 39600
start_tour                  38195
forward_10                  33652
click_progress_bar          27353
backward_10                 25019
click_listen_now            22201
expand_player               21842
next_story                  10162
previous_story               6828
click_more                   6405
click_back                   5174
tour_download_started        4032
click_download_tour          4027
click_location               3630
tour_download_ended          3227
click_3d_map                 1831
click_copy_ref_code           890
cli

lets again run the same checks for different day journeys

In [16]:
listening_events = [
    "start_tour",
    "story_start",
    "story_listened_20",
    "story_listened_40",
    "story_listened_60",
    "story_listened_80",
    "story_completed",
    "play",
    "pause",
    "forward_10",
    "backward_10",
    "click_progress_bar",
    "next_story",
    "previous_story",
    "click_story",
    # "click_item",
    # "click_listen_now"
]

ANDROID_LISTEN = ANDROID_clean[ANDROID_clean["event_name"].isin(listening_events)]

In [17]:
df = ANDROID_LISTEN[["user_pseudo_id", "tour_id", "event_datetime"]].dropna(subset=["tour_id"]).copy()

df["listen_date"] = pd.to_datetime(df["event_datetime"]).dt.date

tour_days = (
    df.groupby(["user_pseudo_id", "tour_id"])["listen_date"]
      .nunique()
      .reset_index(name="distinct_days")
)

multi_day_tours = tour_days[tour_days["distinct_days"] > 1]

print("user-tour pairs total:", len(tour_days))
print("user-tour pairs on multiple days:", len(multi_day_tours))
print("percentage:", len(multi_day_tours) / len(tour_days))

multi_day_tours.head(20)

user-tour pairs total: 9779
user-tour pairs on multiple days: 2037
percentage: 0.20830350751610593


,user_pseudo_id,tour_id,distinct_days
2,000b52af1c5a69eb96f0ef158a18dbe0,535,2
6,003cde637b910a8e8c5e911db3ef41a6,820,2
11,00810b9e3c1156eddc1326d14cc6ca64,869,2
18,00bee13b161139ae8887bdc825a511e0,535,2
20,00c9f1dd8dc43f45b70bef081b0b8571,865,2
21,00d1ca3ccdc87db57b02f3925913f2b7,107,3
23,00d2c86db17217114ba59dac500cf34b,858,2
29,01099eb2970b85d4eb3cc6c34c628800,539,2
30,010cccb789f7ea4ebc32f50ceb9ed292,873,2
32,01245714bf91443f88d5d5997a32e7d6,240,2


In [18]:
multi_day_tours

,user_pseudo_id,tour_id,distinct_days
2,000b52af1c5a69eb96f0ef158a18dbe0,535,2
6,003cde637b910a8e8c5e911db3ef41a6,820,2
11,00810b9e3c1156eddc1326d14cc6ca64,869,2
18,00bee13b161139ae8887bdc825a511e0,535,2
20,00c9f1dd8dc43f45b70bef081b0b8571,865,2
...,...,...,...
9749,ff5182e5c0d01a4bd7b4bae21df79ccb,403,2
9755,ff5e9dc31cb80f688f2886a70904c949,490,2
9758,ff71f5f6260de81efed99cb9746370f9,869,2
9768,ffb2ddc345f0cbc084aa86113e700a58,107,2


In [20]:
weird_guy = ANDROID_clean[ANDROID_clean['user_pseudo_id'] == "605230c80228394702c071059071ca39"]

### Lets check how many of these two day journeys happen because of the time change (at midnight)

In [21]:

pairs = multi_day_tours.loc[multi_day_tours["distinct_days"] == 2, ["user_pseudo_id", "tour_id"]].copy()

df = ANDROID_clean.merge(
    pairs,
    on=["user_pseudo_id", "tour_id"],
    how="inner"
).copy()

df["date"] = df["event_datetime"].dt.date
df["time"] = df["event_datetime"].dt.time

day_bounds = (
    df.groupby(["user_pseudo_id", "tour_id", "date"])["event_datetime"]
      .agg(day_start="min", day_end="max")
      .reset_index()
      .sort_values(["user_pseudo_id", "tour_id", "date"])
)

two_day_windows = (
    day_bounds.groupby(["user_pseudo_id", "tour_id"])
      .agg(
          first_day_end=("day_end", "first"),
          second_day_start=("day_start", "last"),
          n_days=("date", "nunique")
      )
      .reset_index()
)

two_day_windows = two_day_windows[two_day_windows["n_days"] == 2].copy()

two_day_windows["gap_minutes"] = (
    two_day_windows["second_day_start"] - two_day_windows["first_day_end"]
).dt.total_seconds() / 60

midnight_cutoff_cases = two_day_windows[
    (two_day_windows["gap_minutes"] >= 0) &
    (two_day_windows["gap_minutes"] <= 60)
].copy()

print("Two-day user-tour journeys:", len(two_day_windows))
print("Cross-midnight within 1 hour:", len(midnight_cutoff_cases))

midnight_cutoff_cases.head(20)

Two-day user-tour journeys: 1571
Cross-midnight within 1 hour: 2


,user_pseudo_id,tour_id,first_day_end,second_day_start,n_days,gap_minutes
6,01099eb2970b85d4eb3cc6c34c628800,539,2025-08-21 23:59:57.344154,2025-08-22 00:00:05.740,2,0.139931
1708,fa721a6ee44c42449954b18735953575,309,2025-10-23 23:59:52.250165,2025-10-24 00:00:03.082,2,0.180531


User journeys can be defined as tour-pseudo user on the same day. I have not put any inactivity yet, it might be worth to explore it but i dont think this changes much

In [23]:
ANDROID_clean.to_parquet(BASE_DIR / 'data/clean/Android_events.parquet', engine="pyarrow", compression="snappy")